# Z-Image-Turbo GGUF Q6_K — Kaggle T4×2 Guided Demo / Demo hai GPU có hướng dẫn trên Kaggle

## English

This notebook is the public, guided Kaggle entry point for **Z-Image-Turbo GGUF Q6_K on two NVIDIA Tesla T4 GPUs**. It launches the implementation from the [official GitHub repository](https://github.com/dangkhoa2016/Z-Image-Turbo-GGUF-Dual-On-Kaggle-T4x2-GPU); inference and job orchestration remain in the source repository rather than being reimplemented in notebook cells.

The demonstration includes bounded prompt encoding on **GPU0**, independent inference workers on **GPU0 and GPU1**, an SQLite-backed job queue, an authenticated Vue Studio, private 1024×1024 PNG artifacts, and SHA256 receipts. An optional public archive contains **48 quality-gallery PNGs**, **122 benchmark PNGs**, and **40 WebP thumbnails** in two static galleries.

### Before you run

| Requirement | Kaggle setting or resource |
| --- | --- |
| Hardware | **Accelerator: GPU T4 ×2** |
| Internet | **ON** for the initial source, dependency, and gallery downloads |
| Model input | Kaggle Model **dangkhoa2016/unsloth-z-image-turbo-gguf**, GGUF variation **q6-k**, version **1** |
| Support input | Kaggle Dataset **dangkhoa2016/z-image-turbo-diffusers-support-components**, containing the text encoder and VAE |
| Application source | The notebook clones the GitHub release tag **v1.0.0** and prints its resolved commit SHA |

### Recommended execution order

Run **Steps 1–5** in order to prepare the source, verify the GPUs, and start the local services and GPU consumers. **Step 6A** optionally restores the archived image galleries. **Step 6B must run even when Step 6A is skipped**, because it defines the HTTPS tunnel helper required by **Step 7**. Step 7 opens the authenticated Studio, where selecting **Generate** submits a real image job.

A normal **Run All** follows this sequence. **No sample image is generated automatically on notebook startup.** Do not submit confidential prompts through a public demonstration tunnel.

### System overview

~~~text
Browser
  |-- HTTPS tunnel -> Authenticated Vue Studio :18789
  |                     |
  |                  Private API :18788
  |                     |
  |                  SQLite job queue
  |                     |-- GPU0: bounded prompt encoding + worker 0
  |                     '-- GPU1: independent inference worker 1
  |                         -> private PNGs + SHA256
  '-- Separate HTTPS tunnel -> Gallery server :18881
                                |-- 48 quality PNGs
                                '-- 122 benchmark PNGs
~~~

**Verification boundary:** Previous Kaggle runs provide dual-GPU and Studio acceptance evidence, but a complete end-to-end replay of the currently published notebook from a fresh session remains **PENDING**. CPU CI is not a GPU test. Historical diffusion-step measurements are not total job times.

---

## Tiếng Việt

Đây là notebook hướng dẫn công khai để trình diễn **Z-Image-Turbo GGUF Q6_K trên hai GPU NVIDIA Tesla T4 của Kaggle**. Notebook khởi chạy chương trình từ [repository GitHub chính thức](https://github.com/dangkhoa2016/Z-Image-Turbo-GGUF-Dual-On-Kaggle-T4x2-GPU); toàn bộ logic sinh ảnh và điều phối công việc được giữ trong mã nguồn dự án, không viết lại bên trong notebook.

Hệ thống gồm bước mã hóa prompt có giới hạn trên **GPU0**, hai worker inference độc lập trên **GPU0 và GPU1**, hàng đợi công việc SQLite, giao diện Vue Studio có xác thực, ảnh PNG 1024×1024 được lưu riêng tư và biên nhận SHA256. Archive gallery tùy chọn chứa **48 PNG kiểm tra chất lượng**, **122 PNG benchmark** và **40 thumbnail WebP** trong hai trang web tĩnh.

### Chuẩn bị trước khi chạy

| Yêu cầu | Thiết lập hoặc tài nguyên trên Kaggle |
| --- | --- |
| Phần cứng | **Accelerator: GPU T4 ×2** |
| Internet | **BẬT** để tải source, dependency và gallery lần đầu |
| Model | Kaggle Model **dangkhoa2016/unsloth-z-image-turbo-gguf**, biến thể GGUF **q6-k**, phiên bản **1** |
| Thành phần hỗ trợ | Kaggle Dataset **dangkhoa2016/z-image-turbo-diffusers-support-components**, gồm text encoder và VAE |
| Mã nguồn ứng dụng | Notebook clone tag phát hành GitHub **v1.0.0** và in commit SHA thực tế |

### Thứ tự thực hiện

Chạy lần lượt **Bước 1–5** để lấy source, kiểm tra hai GPU, khởi động các dịch vụ nội bộ và GPU consumer. **Bước 6A** chỉ cần thiết nếu muốn khôi phục bộ gallery đã phát hành. **Luôn chạy Bước 6B kể cả khi bỏ qua Bước 6A**, vì Bước 6B tạo hàm mở HTTPS tunnel mà **Bước 7** sử dụng. Tại Bước 7, mở Studio có xác thực và nhấn **Generate** để gửi một job sinh ảnh thực tế.

Chức năng **Run All** thực hiện đúng thứ tự. **Notebook không tự tạo ảnh mẫu khi khởi động.** Không gửi prompt chứa dữ liệu mật qua tunnel trình diễn công khai.

### Tổng quan kiến trúc

~~~text
Trình duyệt
  |-- HTTPS tunnel -> Vue Studio có xác thực :18789
  |                     |
  |                  API nội bộ :18788
  |                     |
  |                  Hàng đợi SQLite
  |                     |-- GPU0: mã hóa prompt có giới hạn + worker 0
  |                     '-- GPU1: worker inference độc lập 1
  |                         -> PNG riêng tư + SHA256
  '-- HTTPS tunnel riêng -> Webserver gallery :18881
                              |-- 48 PNG chất lượng
                              '-- 122 PNG benchmark
~~~

**Giới hạn nghiệm thu:** Các phiên Kaggle trước đã ghi nhận bằng chứng dual-GPU và Studio, nhưng chưa có kết quả **PASS** cho toàn bộ notebook công bố khi chạy lại trên một phiên sạch. Kiểm thử CPU trên CI không thay thế kiểm thử GPU. Thời gian diffusion lịch sử không phải tổng thời gian hoàn thành một job.


## Step 1 — Clone and verify the published source / Bước 1 — Clone và xác minh mã nguồn đã phát hành

### English

**Purpose.** Obtain the application source from GitHub instead of depending on a separate Kaggle dataset that contains a copy of the source. The cell clones the repository into **/kaggle/working/zimage-source-demo** at tag **v1.0.0**. For an existing checkout, the repository's **source_checkout.ensure_source** verifier checks its state without issuing a destructive hard reset.

**Expected evidence.** The output includes **SOURCE_SHA**, **SOURCE_REMOTE_SANITIZED**, **SOURCE_TREE_CLEAN**, and **SOURCE_READY**. Review the resolved commit SHA; importing a notebook from GitHub does not automatically import the rest of the repository. If an existing checkout fails validation, inspect it or choose another workspace rather than deleting user files.

### Tiếng Việt

**Mục đích.** Lấy mã nguồn ứng dụng trực tiếp từ GitHub thay vì phụ thuộc vào một Kaggle dataset khác chứa bản sao source. Cell clone repository vào **/kaggle/working/zimage-source-demo** tại tag **v1.0.0**. Nếu checkout đã tồn tại, hàm **source_checkout.ensure_source** kiểm tra trạng thái mà không chạy lệnh hard reset làm mất thay đổi.

**Kết quả cần thấy.** Output có **SOURCE_SHA**, **SOURCE_REMOTE_SANITIZED**, **SOURCE_TREE_CLEAN** và **SOURCE_READY**. Hãy kiểm tra commit SHA thực tế; thao tác import notebook từ GitHub không tự động tải toàn bộ source của repository. Nếu checkout cũ không hợp lệ, cần kiểm tra hoặc chọn workspace khác thay vì xóa dữ liệu.


In [ ]:
from pathlib import Path
import subprocess, os, sys
REPO_URL="https://github.com/dangkhoa2016/Z-Image-Turbo-GGUF-Dual-On-Kaggle-T4x2-GPU.git"
SOURCE_REF="v1.0.0"       # published release tag
SOURCE_SHA=""           # release tag resolves the source SHA; print/verify the checkout
PROJECT=Path("/kaggle/working/zimage-source-demo")

# Immutable checkout policy: fresh clone if absent; otherwise verify only. Never reset --hard.
if not (PROJECT/".git").is_dir():
    assert not PROJECT.exists(), f"Refusing to overwrite existing project: {PROJECT}"
    subprocess.run(["git","clone","--branch",SOURCE_REF,REPO_URL,str(PROJECT)],check=True)

sys.path.insert(0,str(PROJECT/"scripts"))
try:
    import source_checkout
except Exception as exc:
    raise RuntimeError("Existing checkout lacks scripts/source_checkout.py; clone fresh into a new path instead of resetting.") from exc

info=source_checkout.ensure_source(PROJECT,REPO_URL,source_ref=SOURCE_REF,source_sha=SOURCE_SHA)
print("SOURCE_SHA",info["sha"])
print("SOURCE_REMOTE_SANITIZED",info["remote"])
print("SOURCE_TREE_CLEAN",info["clean"])
assert (PROJECT/"product-studio/index.html").exists()
print("SOURCE_READY",PROJECT)


## Step 2 — Validate model inputs and T4 GPUs / Bước 2 — Kiểm tra đầu vào model và hai GPU T4

### English

**Purpose.** Confirm that Kaggle mounted the exact **Q6_K GGUF** model and the required **text_encoder** and **vae** directories. The cell also checks **nvidia-smi -L** for the two Tesla T4 devices, prints the GGUF file size, and lists installed package versions for diagnosis.

**Expected evidence.** Look for **GPU_T4X2_READY**, **GGUF_BYTES**, and the package-version inventory. If an assertion fails, fix the notebook's Accelerator or attached Inputs before proceeding. Model weights and support components are inference inputs; the application source is still cloned from GitHub.

### Tiếng Việt

**Mục đích.** Xác nhận Kaggle đã mount đúng model **GGUF Q6_K**, cùng hai thư mục **text_encoder** và **vae** bắt buộc. Cell cũng dùng **nvidia-smi -L** để kiểm tra hai Tesla T4, in kích thước file GGUF và liệt kê phiên bản package đang cài nhằm hỗ trợ chẩn đoán.

**Kết quả cần thấy.** Output có **GPU_T4X2_READY**, **GGUF_BYTES** và danh sách phiên bản thư viện. Nếu assertion thất bại, hãy sửa cấu hình Accelerator hoặc các Input đã gắn trước khi tiếp tục. Model và thành phần hỗ trợ là dữ liệu đầu vào inference; source ứng dụng vẫn được clone trực tiếp từ GitHub.


In [ ]:
import subprocess,sys,importlib.metadata as md
model=Path("/kaggle/input/models/dangkhoa2016/unsloth-z-image-turbo-gguf/gguf/q6-k/1/z-image-turbo-Q6_K.gguf")
support=Path("/kaggle/input/datasets/dangkhoa2016/z-image-turbo-diffusers-support-components/z-image-turbo-diffusers-support")
assert model.is_file(),f"Attach Kaggle model first: {model}"
assert (support/"text_encoder").is_dir() and (support/"vae").is_dir(),"Attach support dataset first"
gpus=subprocess.check_output(["nvidia-smi","-L"],text=True)
assert gpus.count("Tesla T4")>=2,"Switch Kaggle accelerator to T4 ×2"
print("GPU_T4X2_READY",gpus)
print("GGUF_BYTES",model.stat().st_size)
for key in ("torch","transformers","diffusers","huggingface-hub","torchao","gguf","safetensors","fastapi","uvicorn"):
    try: print(key,md.version(key))
    except md.PackageNotFoundError:print(key,"MISSING")
print("NOTE: Tensor library versions must match the validated runtime. Install missing packages in the next cell; do not randomly upgrade torch.")


## Step 3 — Verified dependencies and runtime preflight / Bước 3 — Dependency đã kiểm chứng và kiểm tra runtime

### English

**Purpose.** Execute the repository's **scripts/bootstrap_vendor.py** to restore verified upstream runtime packages, then call **scripts/verify_vendor_runtime.py**. The cell configures project-local vendor import paths, checks required Python modules, confirms both CUDA devices, and sets the offline inference flags after the necessary bootstrap work.

**Expected evidence.** The cell prints **PREFLIGHT_READY** with the names of both GPUs. Missing required packages or incompatible dependencies must stop execution. Avoid arbitrarily upgrading Kaggle's Torch installation: a green package-install command alone is not evidence of working inference.

### Tiếng Việt

**Mục đích.** Chạy **scripts/bootstrap_vendor.py** để khôi phục các package runtime upstream đã được kiểm chứng, sau đó chạy **scripts/verify_vendor_runtime.py**. Cell cấu hình đường dẫn import vendor cục bộ, kiểm tra các module Python cần thiết, xác minh hai CUDA device và bật chế độ inference offline sau khi hoàn tất bootstrap.

**Kết quả cần thấy.** Cell in **PREFLIGHT_READY** cùng tên hai GPU. Thiếu thư viện bắt buộc hoặc dependency không tương thích thì phải dừng. Không tự ý nâng cấp Torch trên Kaggle: một lệnh cài package thành công chưa chứng minh được inference hoạt động.


In [ ]:
# Reconstruct audited upstream vendor packages using exact wheel and file-tree SHA256.
# Source remains GitHub; model and support weights remain separately attached inputs.
import os, subprocess, sys, importlib.util
subprocess.run([sys.executable,str(PROJECT/"scripts"/"bootstrap_vendor.py")],check=True)
VENDOR_PATHS=[f"production-system/{name}" for name in
 ("vendor-shim","vendor-hub128","vendor-diffusers041","vendor-torchao015","vendor-gguf")]
assert all((PROJECT / path).is_dir() for path in VENDOR_PATHS)
# Project-local pinned vendor imports take precedence in subprocesses and future kernels.
VENDORS=[str(PROJECT/path) for path in VENDOR_PATHS]
os.environ["PYTHONPATH"]=os.pathsep.join(VENDORS + ([os.environ["PYTHONPATH"]] if os.environ.get("PYTHONPATH") else []))
subprocess.run([sys.executable,str(PROJECT/"scripts"/"verify_vendor_runtime.py")],check=True,env=os.environ.copy())
required=("fastapi","uvicorn","safetensors","transformers","accelerate","httpx","torch")
missing=[package for package in required if importlib.util.find_spec(package) is None]
assert not missing, f"Required runtime packages not installed: {missing}. Provision compatible pinned dependencies before GPU run."
import torch
assert torch.cuda.device_count()==2,"Expected 2 visible T4 GPUs"
os.environ["HF_HUB_OFFLINE"]="1"
os.environ["TRANSFORMERS_OFFLINE"]="1"
print("PREFLIGHT_READY", [torch.cuda.get_device_name(i) for i in range(2)])


## Step 4 — Private API and authenticated Studio / Bước 4 — API nội bộ và Studio có xác thực

### English

**Purpose.** Start the repository's local Studio demonstration, or use its controlled resume path when an existing session is detected. The **private API runs at localhost:18788**, and the authenticated Vue Studio gateway runs at **localhost:18789**. The cell validates both HTTP endpoints and checks the access permissions of locally stored secrets.

**Expected evidence.** **STUDIO_GATEWAY_READY** should appear without printing internal API tokens or service keys. Later HTTPS exposure must target the password-protected gateway on **18789**, never the private API on **18788**.

### Tiếng Việt

**Mục đích.** Khởi động Studio demo của repository hoặc dùng luồng khôi phục có kiểm soát khi phát hiện phiên đã tồn tại. **API nội bộ chạy tại localhost:18788**, còn gateway Vue Studio có xác thực chạy tại **localhost:18789**. Cell kiểm tra hai endpoint HTTP và quyền truy cập file chứa thông tin bí mật cục bộ.

**Kết quả cần thấy.** Xuất hiện **STUDIO_GATEWAY_READY** mà không in token API hoặc service key nội bộ. Khi mở HTTPS ở bước sau, chỉ công khai gateway có mật khẩu ở cổng **18789**, tuyệt đối không công khai API riêng tư ở cổng **18788**.


In [ ]:
import secrets,json,time,urllib.request,shutil
from pathlib import Path
PROD=PROJECT/"production-system"
RUN=PROD/"b3-studio-demo-live-20261016"
cfgpath=RUN/"local-secrets.json"
if not RUN.exists():
    subprocess.run([sys.executable,str(PROD/"studio-demo/start_local_demo.py")],check=True,env=os.environ.copy())
else:
    assert cfgpath.exists(),"Incomplete runtime: manual inspection required"
    subprocess.run([sys.executable,str(PROJECT/"deployment"/"resume_local_demo.py")],check=True,env=os.environ.copy())
cfg=json.loads(cfgpath.read_text())
assert all(cfg.get(key) for key in ("api_token","studio_password","service_key"))
assert cfgpath.stat().st_mode & 0o077==0,"Unsafe secret permissions"
for port,path in ((18788,"/health"),(18789,"/auth/session")):
    with urllib.request.urlopen(f"http://127.0.0.1:{port}{path}",timeout=10) as res:
        assert res.status==200
print("STUDIO_GATEWAY_READY: http://127.0.0.1:18789/studio/")
print("Service keys are NOT printed")


## Step 5 — Start the dual-GPU job consumers / Bước 5 — Khởi động consumer cho hai GPU

### English

**Purpose.** Validate the SQLite job database and report queue states before starting the persistent GPU consumer. If the expected consumer is already running, the cell reuses that state rather than launching a duplicate. **GPU0 handles bounded prompt encoding as well as inference worker 0**; **GPU1 hosts inference worker 1**. Independent workers enable multiple jobs but do not guarantee that an individual image renders twice as fast.

**Expected evidence.** The output includes **QUEUE_STATE** and **T4×2 DISPATCH READY**. This step **does not create a sample image**. Use the authenticated Studio in Step 7 to submit different prompts and examine per-job GPU assignment and PNG receipts.

### Tiếng Việt

**Mục đích.** Xác minh tính toàn vẹn của cơ sở dữ liệu SQLite và thống kê hàng đợi trước khi khởi động GPU consumer thường trú. Nếu consumer đúng đã chạy, cell tái sử dụng trạng thái đó thay vì tạo tiến trình trùng lặp. **GPU0 vừa mã hóa prompt có giới hạn vừa chạy inference worker 0**; **GPU1 chạy inference worker 1**. Hai worker độc lập giúp xử lý nhiều job, nhưng không bảo đảm một ảnh sẽ sinh nhanh gấp đôi.

**Kết quả cần thấy.** Output có **QUEUE_STATE** và **T4×2 DISPATCH READY**. Bước này **chưa tạo ảnh mẫu**. Hãy sử dụng Studio có xác thực ở Bước 7 để gửi các prompt khác nhau, sau đó kiểm tra GPU được phân công và biên nhận PNG của từng job.


In [ ]:
import sqlite3
with sqlite3.connect("file:"+str(RUN/"api.sqlite3")+"?mode=ro",uri=True) as db:
    assert db.execute("PRAGMA integrity_check").fetchone()[0]=="ok"
    print("QUEUE_STATE",list(db.execute("select state,count(*) from jobs group by state")))
consumer=subprocess.run([sys.executable,str(PROJECT/"deployment"/"start_persistent_gpu_consumer.py")],capture_output=True,text=True,env=os.environ.copy())
# Existing worker means idempotent for the notebook rerun; do not spawn duplicate.
if consumer.returncode:
    if "Existing persistent consumer PID" not in (consumer.stdout+consumer.stderr):
        raise RuntimeError((consumer.stdout+consumer.stderr)[-1300:])
    print("GPU consumer already running; keeping existing process")
else:
    print(consumer.stdout.strip())
print("T4×2 DISPATCH READY: new Studio submissions are handled by the existing lifecycle.")
print("GPU 0 encodes prompts in a bounded phase; independent inference workers can occupy both GPU 0 and GPU 1.")


## Step 6A — Restore the 170-image Release galleries (optional) / Bước 6A — Khôi phục gallery 170 ảnh (tùy chọn)

### English

**Purpose.** Retrieve the public **zimage-gallery-170-watermarked-public-v1.0.0.tar.gz** from GitHub Release. The cell checks the pinned SHA256, safely extracts the archive, validates its file manifest, and verifies exact counts: **48 quality PNGs**, **122 benchmark PNGs**, and **40 WebP thumbnails**. After Step 6B, open two independent HTTPS domains directly at their root. Local ports are 18881 (quality) and 18882 (benchmark).. When verification passes, the two static gallery servers run at **localhost:18881** and **localhost:18882**.

**Expected evidence.** **GALLERY_170_SHA256_VERIFIED** and **GALLERY_SERVER_READY** mean the archive passed its integrity checks and can be served. If unavailable, **OPTIONAL_GALLERY_170_UNAVAILABLE** or **GALLERY_SERVER_SKIPPED** is reported without treating it as a GPU failure. These are historical archived images; this cell **does not render them**.

### Tiếng Việt

**Mục đích.** Tải file **zimage-gallery-170-watermarked-public-v1.0.0.tar.gz** từ GitHub Release. Cell xác minh SHA256 đã ghim, giải nén an toàn, đối chiếu manifest và kiểm tra chính xác **48 PNG chất lượng**, **122 PNG benchmark**, **40 thumbnail WebP**. Sau Bước 6B, mở trực tiếp trang chủ của hai tên miền HTTPS độc lập. Hai cổng nội bộ là 18881 (quality) và 18882 (benchmark).. Khi tất cả điều kiện đều đạt, hai webserver gallery tĩnh chạy ở **localhost:18881** và **localhost:18882**.

**Kết quả cần thấy.** **GALLERY_170_SHA256_VERIFIED** và **GALLERY_SERVER_READY** cho biết archive đã vượt qua bước kiểm tra toàn vẹn và có thể phục vụ qua webserver. Nếu không khả dụng, cell báo **OPTIONAL_GALLERY_170_UNAVAILABLE** hoặc **GALLERY_SERVER_SKIPPED** mà không coi đây là lỗi GPU. Đây là những ảnh đã tạo từ trước; cell **không tự render ảnh**.


In [ ]:
import hashlib, tarfile, shutil, urllib.request, http.server, socketserver, threading
from pathlib import Path
GALLERY_NAME="zimage-gallery-170-watermarked-public-v1.0.0.tar.gz"
EXPECTED_SHA256="e0240d7bc60314f9e8394b5afd922a330216ecd6b596122e381f0bab3e755997"
URL="https://github.com/dangkhoa2016/Z-Image-Turbo-GGUF-Dual-On-Kaggle-T4x2-GPU/releases/download/v1.0.0/"+GALLERY_NAME
ARCHIVE=Path("/kaggle/working")/GALLERY_NAME
GROOT=Path("/kaggle/working/zimage-gallery-showcase")
STAGE=Path("/kaggle/working/.zimage-gallery-170-staging")
ready={}
def file_sha256(path):
    h=hashlib.sha256()
    with path.open("rb") as f:
        for piece in iter(lambda:f.read(1024*1024),b""): h.update(piece)
    return h.hexdigest()
def restore_approved_galleries():
    if not ARCHIVE.is_file() or file_sha256(ARCHIVE)!=EXPECTED_SHA256:
        urllib.request.urlretrieve(URL,ARCHIVE)
    assert file_sha256(ARCHIVE)==EXPECTED_SHA256,"Gallery archive SHA256 mismatch"
    if not GROOT.exists():
        assert not STAGE.exists(),"Incomplete staging directory"
        STAGE.mkdir()
        try:
            with tarfile.open(ARCHIVE,"r:gz") as tf:
                for member in tf:
                    assert member.isfile() or member.isdir(),"Unsupported TAR entry"
                    dest=(STAGE/member.name).resolve()
                    assert dest.is_relative_to(STAGE.resolve()),"Unsafe TAR path"
                    if member.isdir():dest.mkdir(parents=True,exist_ok=True)
                    else:
                        dest.parent.mkdir(parents=True,exist_ok=True)
                        with tf.extractfile(member) as src,dest.open("wb") as dst:shutil.copyfileobj(src,dst)
            extracted=STAGE/"zimage-gallery-170"
            assert extracted.is_dir() and not GROOT.exists()
            extracted.rename(GROOT)
        finally: shutil.rmtree(STAGE,ignore_errors=True)
    checksums=(GROOT/"SHA256SUMS.txt").read_text().splitlines()
    assert len(checksums)>170,"Incomplete gallery checksum manifest"
    for line in checksums:
        expected,rel=line.split("  ",1)
        path=(GROOT/rel).resolve()
        assert path.is_relative_to(GROOT.resolve()) and path.is_file(),"Missing gallery asset: "+rel
        assert file_sha256(path)==expected,"Gallery checksum mismatch: "+rel
    assert len(list((GROOT/"gallery-48/images").glob("*.png")))==48
    assert len(list((GROOT/"gallery-122/output").rglob("*.png")))==122
    assert len(list((GROOT/"gallery-48/thumbs").glob("*.webp")))==40
    assert (GROOT/"gallery-48/index.html").is_file()
    assert (GROOT/"gallery-122/all-122-gallery/index.html").is_file()
    ready.update({"quality":18881,"benchmark":18882})
    print("GALLERY_170_SHA256_VERIFIED: 48 quality PNG + 122 benchmark PNG + 40 WebP thumbnails")
try:
    restore_approved_galleries()
except Exception as exc:
    ready.clear()
    print("OPTIONAL_GALLERY_170_UNAVAILABLE",type(exc).__name__,str(exc)[:250])
if ready:
    from urllib.parse import urlsplit
    from functools import partial
    gallery_urls={}
    class GalleryHandler(http.server.SimpleHTTPRequestHandler):
        ROOTS={"quality":GROOT/"gallery-48","benchmark":GROOT/"gallery-122"}
        def __init__(self,*args,gallery_kind,**kwargs):
            self.gallery_kind=gallery_kind
            super().__init__(*args,directory=str(self.ROOTS[gallery_kind]),**kwargs)
        def translate_path(self,path):
            route=urlsplit(path).path
            if self.gallery_kind=="benchmark" and not route.startswith("/output/"):
                route="/all-122-gallery"+route
            return super().translate_path(route)
        def do_GET(self):
            route=urlsplit(self.path).path
            if route in ("/","/index.html"):
                rel="index.html" if self.gallery_kind=="quality" else "all-122-gallery/index.html"
                html=(self.ROOTS[self.gallery_kind]/rel).read_text(encoding="utf-8")
                html=html.replace("/gallery-48/index.html",gallery_urls.get("quality","/"))
                html=html.replace("/gallery-122/all-122-gallery/index.html",gallery_urls.get("benchmark","/"))
                body=html.encode("utf-8")
                self.send_response(200)
                self.send_header("Content-Type","text/html; charset=utf-8")
                self.send_header("Content-Length",str(len(body)))
                self.end_headers()
                self.wfile.write(body)
            else:
                return super().do_GET()
    class ReusableServer(socketserver.ThreadingTCPServer):
        allow_reuse_address=True
    gallery_servers={}
    for kind,port in (("quality",18881),("benchmark",18882)):
        server=ReusableServer(("127.0.0.1",port),partial(GalleryHandler,gallery_kind=kind))
        threading.Thread(target=server.serve_forever,daemon=True).start()
        gallery_servers[kind]=server
        assert urllib.request.urlopen("http://127.0.0.1:"+str(port)+"/",timeout=5).status==200
    print("GALLERY_SERVERS_READY quality=18881 benchmark=18882")
else:
    print("GALLERY_SERVERS_SKIPPED")


## Step 6B — Prepare HTTPS and publish the optional galleries / Bước 6B — Chuẩn bị HTTPS và tùy chọn công khai gallery

### English

**Purpose.** Prepare the **cloudflared** executable and define **quick_tunnel(name, port)** for both galleries and the Studio. When Step 6A restored the galleries, this cell opens two independent temporary HTTPS domains, each serving its gallery at the domain root.

**Execution requirement.** **Run Step 6B even if you skipped Step 6A**, because Step 7 calls the tunnel helper defined here. Without gallery files, this step simply prepares the helper and reports **GALLERY_TUNNEL_SKIPPED**.

**Expected evidence.** Two root HTTPS addresses, **QUALITY_GALLERY_HTTPS** and **BENCHMARK_GALLERY_HTTPS**, or a clear skip message. They remain separate from the authenticated Studio. A working tunnel is **not** evidence that a new GPU image job completed.

### Tiếng Việt

**Mục đích.** Chuẩn bị chương trình **cloudflared** và định nghĩa hàm **quick_tunnel(name, port)** cho hai gallery và Studio. Nếu Bước 6A khôi phục thành công, cell mở hai tên miền HTTPS tạm thời độc lập, mỗi gallery truy cập tại trang chủ của domain.

**Yêu cầu thứ tự.** **Luôn chạy Bước 6B dù đã bỏ qua Bước 6A**, bởi Bước 7 gọi hàm tạo tunnel được khai báo ở đây. Khi không có dữ liệu gallery, cell vẫn chuẩn bị hàm cần thiết và chỉ in **GALLERY_TUNNEL_SKIPPED**.

**Kết quả cần thấy.** Có hai địa chỉ **QUALITY_GALLERY_HTTPS** và **BENCHMARK_GALLERY_HTTPS**, hoặc thông báo bỏ qua rõ ràng. Hai tunnel độc lập với Studio có xác thực. Tunnel hoạt động **không chứng minh** đã có job GPU mới hoàn thành.


In [ ]:
import shutil,urllib.request,re,subprocess,time
from pathlib import Path
cf=Path("/kaggle/working/cloudflared")
if not cf.exists():
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",cf)
    cf.chmod(0o700)
def quick_tunnel(name,port):
    logfile=Path(f"/kaggle/working/{name}-cloudflared.log")
    proc=subprocess.Popen([str(cf),"tunnel","--no-autoupdate","--url",f"http://127.0.0.1:{port}"],stdout=logfile.open("ab"),stderr=subprocess.STDOUT,start_new_session=True)
    print("TUNNEL_PID",proc.pid)
    for attempt in range(50):
        text=logfile.read_text(errors="replace")
        urls=re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com",text)
        if urls:return urls[-1]
        if proc.poll() is not None:raise RuntimeError(f"{name} tunnel exited: {text[-1200:]}")
        time.sleep(1)
    raise RuntimeError(f"Tunnel URL not found in {logfile}")
ready=globals().get("ready",{})
if ready:
    gallery_urls["quality"]=quick_tunnel("gallery-quality",18881)
    gallery_urls["benchmark"]=quick_tunnel("gallery-benchmark",18882)
    print("QUALITY_GALLERY_HTTPS:",gallery_urls["quality"]+"/")
    print("BENCHMARK_GALLERY_HTTPS:",gallery_urls["benchmark"]+"/")
else:
    print("GALLERY_TUNNELS_SKIPPED: gallery archive unavailable")


## Step 7 — Open the authenticated Studio over HTTPS / Bước 7 — Mở Studio có xác thực qua HTTPS

### English

**Purpose.** Confirm the Studio gateway is healthy, open a separate Cloudflare Quick Tunnel to **localhost:18789**, and print a disposable demonstration URL, username, and password. The notebook intentionally displays **demo login credentials only**; internal service tokens must remain private.

**How to use the Studio.** Open the printed **/studio/** URL, sign in, submit your own prompt, observe queued/running/completed states, and review the generated image and elapsed seconds. For dual-GPU evidence, submit **two distinguishable prompts** and inspect both job receipts and worker assignments.

**Expected evidence.** A **Z-IMAGE STUDIO DEMO** block containing **URL**, **Username**, and **Password**. This step publishes an authenticated interface; it **does not generate an image by itself**. Do not enter sensitive content in a publicly reachable demonstration.

### Tiếng Việt

**Mục đích.** Xác minh gateway Studio, mở Cloudflare Quick Tunnel riêng tới **localhost:18789**, rồi in URL, tên người dùng và mật khẩu demo dùng tạm. Notebook chỉ cố ý hiển thị **thông tin đăng nhập demo**; token dịch vụ nội bộ luôn phải được giữ riêng tư.

**Cách dùng Studio.** Mở URL **/studio/** được in ra, đăng nhập, gửi prompt, theo dõi các trạng thái queued/running/completed rồi xem ảnh và thời gian xử lý theo giây. Để thu bằng chứng hai GPU, gửi **hai prompt phân biệt** và kiểm tra biên nhận cùng worker của từng job.

**Kết quả cần thấy.** Khối **Z-IMAGE STUDIO DEMO** chứa **URL**, **Username** và **Password**. Bước này chỉ công khai giao diện có xác thực; **không tự tạo ảnh**. Không nhập nội dung nhạy cảm vào môi trường demo có thể truy cập công khai.


In [ ]:
assert urllib.request.urlopen("http://127.0.0.1:18789/auth/session",timeout=8).status==200
studio_url=quick_tunnel("studio",18789)
print("======================================")
print("Z-IMAGE STUDIO DEMO")
print("URL:      ",studio_url+"/studio/")
print("Username:  studio")
print("Password: ",cfg["studio_password"])
print("======================================")
print("This is a publicly reachable demo login; do not use it for sensitive workloads.")


## Step 8 — Review acceptance and protect session outputs / Bước 8 — Đánh giá nghiệm thu và bảo vệ output phiên chạy

### English

**Acceptance checklist**

1. **Source:** Confirm **SOURCE_SHA**, **SOURCE_TREE_CLEAN**, and the intended Git revision.
2. **Hardware and runtime:** Confirm **GPU_T4X2_READY** and **PREFLIGHT_READY** on both Tesla T4 devices.
3. **Local services:** Confirm **STUDIO_GATEWAY_READY** and **T4×2 DISPATCH READY**, without disclosing internal service tokens.
4. **Fresh GPU verification:** Submit distinguishable prompts through Studio. Inspect two completed 1024×1024 PNGs, worker/GPU assignments, SHA256 receipts, prompt fidelity, and measured elapsed seconds. **Do not mark the clean-session notebook PASS until this has actually been observed.**
5. **Optional gallery:** **GALLERY_170_SHA256_VERIFIED** establishes archive integrity, not universal visual-quality or prompt-fidelity acceptance.

**Timing interpretation:** Queueing, setup, prompt encoding, diffusion and saving are distinct phases. Historical **nine-step diffusion timings** are not the complete elapsed time of an image job.

**Before shutting down:** Save the generated artifacts and any review logs you want to retain. Kaggle runtime state and Cloudflare Quick Tunnel links are temporary. The original unwatermarked gallery images are not included in the public release.

**Further reading:** [English README](https://github.com/dangkhoa2016/Z-Image-Turbo-GGUF-Dual-On-Kaggle-T4x2-GPU/blob/main/README.md) · [Evidence index](https://github.com/dangkhoa2016/Z-Image-Turbo-GGUF-Dual-On-Kaggle-T4x2-GPU/blob/main/docs/evidence-index.md) · [Release v1.0.0](https://github.com/dangkhoa2016/Z-Image-Turbo-GGUF-Dual-On-Kaggle-T4x2-GPU/releases/tag/v1.0.0).

---

### Tiếng Việt

**Danh sách nghiệm thu**

1. **Mã nguồn:** Kiểm tra **SOURCE_SHA**, **SOURCE_TREE_CLEAN** và phiên bản Git đúng yêu cầu.
2. **GPU và runtime:** Xác nhận **GPU_T4X2_READY**, **PREFLIGHT_READY** trên đúng hai Tesla T4.
3. **Dịch vụ nội bộ:** Xác nhận **STUDIO_GATEWAY_READY**, **T4×2 DISPATCH READY**, không để lộ token nội bộ.
4. **Kiểm thử GPU mới:** Gửi các prompt khác nhau qua Studio. Kiểm tra hai PNG 1024×1024 đã hoàn thành, GPU/worker được phân công, SHA256, mức độ bám sát prompt và số giây đã đo. **Chỉ đánh dấu PASS cho notebook trên phiên sạch sau khi quan sát được đầy đủ các kết quả này.**
5. **Gallery tùy chọn:** **GALLERY_170_SHA256_VERIFIED** chỉ chứng minh archive hợp lệ, không có nghĩa tất cả ảnh đều đạt yêu cầu chất lượng và ngữ nghĩa.

**Cách hiểu thời gian:** Chờ hàng đợi, chuẩn bị, mã hóa prompt, diffusion và lưu ảnh là các giai đoạn riêng. Số giây **diffusion 9 bước** trong báo cáo cũ không phải thời gian hoàn thành toàn bộ một job.

**Trước khi tắt máy:** Lưu lại những ảnh được sinh ra và các log cần thiết. Trạng thái runtime Kaggle và đường dẫn Cloudflare Quick Tunnel chỉ tồn tại tạm thời. Bộ ảnh gallery gốc chưa watermark không nằm trong bản phát hành công khai.

**Tài liệu liên quan:** [README tiếng Việt](https://github.com/dangkhoa2016/Z-Image-Turbo-GGUF-Dual-On-Kaggle-T4x2-GPU/blob/main/README.vi.md) · [Chỉ mục bằng chứng](https://github.com/dangkhoa2016/Z-Image-Turbo-GGUF-Dual-On-Kaggle-T4x2-GPU/blob/main/docs/evidence-index.vi.md) · [Release v1.0.0](https://github.com/dangkhoa2016/Z-Image-Turbo-GGUF-Dual-On-Kaggle-T4x2-GPU/releases/tag/v1.0.0).
